<a href="https://colab.research.google.com/github/Sk-Kamrej/Bank_Marketing_Project/blob/main/BMP_ADASYN_Advance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================
#  BANK MARKETING PROJECT (Machine Learning)
# =============================================================

In [4]:
# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================

!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [14]:
# ==============================
# 2. IMPORTS LIBRARIES
# ==============================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN

In [6]:
# ==============================
# 3. LOAD DATASET
# ==============================

df = pd.read_csv('/content/Bank_Marketing_Full.csv', sep=';')

In [7]:
# ==============================
# 4. PREPROCESSING
# ==============================

# Drop Leakage Feature
df = df.drop('duration', axis=1)

# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [8]:
# ==============================
# 5. SPLIT DATA
# ==============================

# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ADASYN Oversampling
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)

In [9]:
# =========================================
# 6. CUSTOM METRICS: SPECIFICITY & G-MEAN
# =========================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    spec = tn / (tn + fp)
    return np.sqrt(sensitivity * spec)

In [10]:
# ==============================
# 7. EVALUATION METRICS
# ==============================
from sklearn.metrics import make_scorer

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': make_scorer(specificity),
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc': make_scorer(matthews_corrcoef),
    'g_mean': make_scorer(gmean),
    'kappa': make_scorer(cohen_kappa_score)
}

In [11]:
# ==============================
# 8. METRICS FUNCTION
# ==============================

def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

In [12]:
# ==============================
# 9. CROSS VALIDATION
# ==============================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [28]:
# ==============================
# MODEL: LogisticRegression
# ==============================
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', LogisticRegression(max_iter=500))
])

params = {
    "model__C": [0.01, 0.1, 0.6, 0.8, 1],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear", "saga"]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Logistic Regression")
compute_metrics(y_test, y_pred, y_prob)

Model: Logistic Regression
Accuracy    : 0.8780
Precision   : 0.4579
Recall      : 0.4515
Specificity : 0.9321
F1 Score    : 0.4547
ROC-AUC     : 0.7761
MCC         : 0.3860
G-Mean      : 0.6487
Kappa       : 0.3860


In [32]:
# ==============================
# MODEL: Decision Tree
# ==============================
from sklearn.tree import DecisionTreeClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', DecisionTreeClassifier(class_weight='balanced'))
])

params = {
    "model__max_depth": [5, 8, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Decision Tree")
compute_metrics(y_test, y_pred, y_prob)

Model: Decision Tree
Accuracy    : 0.8705
Precision   : 0.4398
Recall      : 0.5474
Specificity : 0.9115
F1 Score    : 0.4878
ROC-AUC     : 0.7648
MCC         : 0.4179
G-Mean      : 0.7064
Kappa       : 0.4146


In [38]:
# ==============================
# MODEL: Random Forest
# ==============================
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', RandomForestClassifier())
])

params = {
    "model__n_estimators":[100,200,300],
    "model__max_depth":[10,15,20],
    "model__min_samples_split":[5,10],
    "model__min_samples_leaf":[2,4],
    "model__max_features":["sqrt"],
    "model__criterion":["entropy"]
}
model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Random Forest")
compute_metrics(y_test, y_pred, y_prob)

Model: Random Forest
Accuracy    : 0.8840
Precision   : 0.4863
Recall      : 0.5345
Specificity : 0.9283
F1 Score    : 0.5092
ROC-AUC     : 0.8051
MCC         : 0.4442
G-Mean      : 0.7044
Kappa       : 0.4436


In [42]:
# ==============================
# MODEL: Gradient Boosting
# ==============================
from sklearn.ensemble import GradientBoostingClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', GradientBoostingClassifier())
])

params = {
    "model__n_estimators":[200,300],
    "model__learning_rate":[0.03,0.05,0.1]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Gradient Boosting")
compute_metrics(y_test, y_pred, y_prob)

Model: Gradient Boosting
Accuracy    : 0.8878
Precision   : 0.5025
Recall      : 0.4407
Specificity : 0.9446
F1 Score    : 0.4696
ROC-AUC     : 0.7925
MCC         : 0.4083
G-Mean      : 0.6452
Kappa       : 0.4072


In [47]:
# ==============================
# MODEL: XGBoost
# ==============================
from xgboost import XGBClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', XGBClassifier(eval_metric='logloss', use_label_encoder=False))
])

params = {
    "model__n_estimators":[300,500,700],
    "model__learning_rate":[0.01,0.03,0.05],
    "model__max_depth":[3,4,5],
    "model__subsample":[0.8,1.0],
    "model__colsample_bytree":[0.8,1.0],
    "model__gamma":[0,1],
    "model__reg_lambda":[1,2]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: XGBoost")
compute_metrics(y_test, y_pred, y_prob)

Model: XGBoost
Accuracy    : 0.8866
Precision   : 0.4966
Recall      : 0.4709
Specificity : 0.9394
F1 Score    : 0.4834
ROC-AUC     : 0.7966
MCC         : 0.4200
G-Mean      : 0.6651
Kappa       : 0.4198


In [50]:
# ==============================
# MODEL: LightGBM
# ==============================
from lightgbm import LGBMClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', LGBMClassifier(class_weight='balanced', verbosity=-1))
])

params = {
    "model__n_estimators":[300,500],
    "model__learning_rate":[0.03,0.05],
    "model__num_leaves":[31,63,127]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='recall', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: LightGBM")
compute_metrics(y_test, y_pred, y_prob)

Model: LightGBM
Accuracy    : 0.8818
Precision   : 0.4785
Recall      : 0.5517
Specificity : 0.9237
F1 Score    : 0.5125
ROC-AUC     : 0.8000
MCC         : 0.4471
G-Mean      : 0.7139
Kappa       : 0.4456


In [77]:
# ==============================
# MODEL: CatBoost
# ==============================
from catboost import CatBoostClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.8, random_state=42)),
    ('model', CatBoostClassifier(verbose=0))
])

params = {
    "model__iterations":[200,300,400],
    "model__depth":[4,6,8],
    "model__learning_rate":[0.03,0.05]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: CatBoost")
compute_metrics(y_test, y_pred, y_prob)

Model: CatBoost
Accuracy    : 0.8867
Precision   : 0.4970
Recall      : 0.4429
Specificity : 0.9431
F1 Score    : 0.4684
ROC-AUC     : 0.7913
MCC         : 0.4061
G-Mean      : 0.6463
Kappa       : 0.4052


In [72]:
# ==============================
# MODEL: SVM
# ==============================
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', CalibratedClassifierCV(estimator=LinearSVC(max_iter=1000, random_state=42)))
])

params = {"model__estimator__C":[0.1,0.5,1]}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: SVM")
compute_metrics(y_test, y_pred, y_prob)

Model: SVM
Accuracy    : 0.8769
Precision   : 0.4476
Recall      : 0.3955
Specificity : 0.9380
F1 Score    : 0.4199
ROC-AUC     : 0.7555
MCC         : 0.3522
G-Mean      : 0.6091
Kappa       : 0.3514


In [81]:
# ==============================
# MODEL: KNN
# ==============================
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.3, random_state=42)),
    ('model', KNeighborsClassifier())
])

params = {
    "model__n_neighbors":[5,7,9],
    "model__weights":['uniform','distance']
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: KNN")
compute_metrics(y_test, y_pred, y_prob)

Model: KNN
Accuracy    : 0.8736
Precision   : 0.4411
Recall      : 0.4558
Specificity : 0.9267
F1 Score    : 0.4483
ROC-AUC     : 0.7616
MCC         : 0.3771
G-Mean      : 0.6499
Kappa       : 0.3770


In [87]:
# ==============================
# MODEL: MLP
# ==============================
from sklearn.neural_network import MLPClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.2, random_state=42)),
    ('model', MLPClassifier(max_iter=300, alpha=0.001, early_stopping=True))
])

params = {
    "model__hidden_layer_sizes":[(50,), (100,), (50,50)],
    "model__alpha":[0.0001, 0.001, 0.01]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='roc_auc', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: MLP")
compute_metrics(y_test, y_pred, y_prob)

Model: MLP
Accuracy    : 0.8803
Precision   : 0.4669
Recall      : 0.4407
Specificity : 0.9361
F1 Score    : 0.4534
ROC-AUC     : 0.7871
MCC         : 0.3865
G-Mean      : 0.6423
Kappa       : 0.3863


In [93]:
# ==============================
# MODEL: Bagging
# ==============================
from sklearn.ensemble import BaggingClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.3, random_state=42)),
    ('model', BaggingClassifier())
])

params = {"model__n_estimators":[50,100]}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='roc_auc', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Bagging")
compute_metrics(y_test, y_pred, y_prob)

Model: Bagging
Accuracy    : 0.8871
Precision   : 0.4985
Recall      : 0.3642
Specificity : 0.9535
F1 Score    : 0.4209
ROC-AUC     : 0.7685
MCC         : 0.3655
G-Mean      : 0.5893
Kappa       : 0.3601


In [96]:
# ==============================
# MODEL: Stacking
# ==============================
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=200)),
        ('dt', DecisionTreeClassifier(max_depth=10)),
        ('xgb', XGBClassifier(eval_metric='logloss', use_label_encoder=False)),
        ('lgbm', LGBMClassifier()),
        ('cat', CatBoostClassifier(verbose=0)),
        ('gb', GradientBoostingClassifier())
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    cv=3,
    n_jobs=-1
)


pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.3, random_state=42)),
    ('model', stacking_model)
])

model = pipe.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: Stacking")
compute_metrics(y_test, y_pred, y_prob)

Model: Stacking
Accuracy    : 0.8917
Precision   : 0.5280
Recall      : 0.3664
Specificity : 0.9584
F1 Score    : 0.4326
ROC-AUC     : 0.7796
MCC         : 0.3825
G-Mean      : 0.5926
Kappa       : 0.3749
